# TraceLog Benchmark v3.2 — Results

## Background

**v3.2** is a model lineup refresh of v3:

| Provider | v3 model | v3.2 model |
|---|---|---|
| OpenAI | `gpt-4o` | `gpt-5.4` |
| Anthropic | `claude-opus-4-6` | `claude-sonnet-4-6` |
| Google | `gemini-2.5-pro` | `gemini-2.5-pro` (unchanged) |

Same 7 scenarios, same 2 conditions, same benchmark framework as v3. Objective: measure whether newer models show different TraceLog sensitivity.

Result structure: `scenario × provider × condition(A|B)` — 7 × 3 × 2 = **42 agent runs**

---

## Conditions

**Condition A — Standard Log + Agent**: Standard Python log output.

**Condition B — TraceLog + Agent**: Trace-DSL dump with causal chain. On each failed `write_file`, the tool re-runs the scenario with TraceLog and returns an updated trace.

---

## Note on Google producer_aggregator B

`GraphRecursionError` (recursion_limit=30) on two consecutive runs. Recorded as fix_success=False. Not a transient API issue — Gemini consistently fails to converge on this scenario with TraceLog context.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

NOTEBOOK_DIR = Path(os.path.abspath('__file__')).parent
RUNS_DIR = Path('docs/eval/benchmark_v3.2/runs')

PROVIDERS = ['openai', 'google', 'anthropic']
PROVIDER_LABELS = {
    'openai':    'OpenAI gpt-5.4',
    'google':    'Google gemini-2.5-pro',
    'anthropic': 'Anthropic claude-sonnet-4-6',
}
SCENARIOS_V2 = ['api_gateway', 'maze', 'dynamic_pricing', 'thread_local']
SCENARIOS_V3 = ['worker_dispatch', 'producer_aggregator', 'ledger_processor']
ALL_SCENARIOS = SCENARIOS_V2 + SCENARIOS_V3

COLOR_A = '#E07B7B'
COLOR_B = '#6BAED6'

# Load results — latest run per (scenario, provider)
raw = {}
for run_dir in sorted(RUNS_DIR.iterdir()):
    if run_dir.name.startswith('_exec'):
        continue
    result_path = run_dir / 'result.json'
    if not result_path.exists():
        continue
    r = json.loads(result_path.read_text())
    key = (r['scenario'], r['provider'])
    if key not in raw or r['generated_at'] > raw[key]['generated_at']:
        raw[key] = r

results = list(raw.values())
print(f'Loaded {len(results)} runs ({len(PROVIDERS)} providers × {len(ALL_SCENARIOS)} scenarios)')
for s in ALL_SCENARIOS:
    ps = sorted(r['provider'] for r in results if r['scenario'] == s)
    print(f'  {s}: {ps}')

In [ ]:
# Build flat DataFrame
rows = []
for r in results:
    for cond in ['A', 'B']:
        d = r[cond]
        rows.append({
            'scenario':      r['scenario'],
            'provider':      r['provider'],
            'model':         r['model'],
            'scenario_type': 'multi-threading' if r['scenario'] in SCENARIOS_V3 else 'single-process',
            'condition':     cond,
            'fix_success':   d['fix_success'],
            'root_cause':    d.get('root_cause_identified', False),
            'tool_calls':    d['tool_call_count'],
            'fix_attempts':  d['fix_attempts'],
            'iterations':    d['iterations'],
            'latency':       d['latency'],
            'total_tokens':  d['usage']['total_tokens'],
        })

df = pd.DataFrame(rows)
df

## Provider-Level Aggregate

In [ ]:
agg_rows = []
for p in PROVIDERS:
    sub = df[df['provider'] == p]
    for cond in ['A', 'B']:
        c = sub[sub['condition'] == cond]
        agg_rows.append({
            'provider':  PROVIDER_LABELS[p],
            'condition': f'{cond} — {"Standard Log" if cond == "A" else "TraceLog"}',
            'fix_rate':        c['fix_success'].mean(),
            'avg_tools':       c['tool_calls'].mean(),
            'avg_attempts':    c['fix_attempts'].mean(),
            'avg_latency':     c['latency'].mean(),
            'avg_tokens':      c['total_tokens'].mean(),
        })

agg_df = pd.DataFrame(agg_rows).set_index(['provider', 'condition'])
agg_df.columns = ['Fix Rate', 'Avg Tools', 'Avg Attempts', 'Avg Latency (s)', 'Avg Tokens']
agg_df.round(2)

## TraceLog Effect: B vs A Delta

In [ ]:
delta_rows = []
for p in PROVIDERS:
    sub = df[df['provider'] == p]
    a = sub[sub['condition'] == 'A']
    b = sub[sub['condition'] == 'B']

    def pct(col):
        av, bv = a[col].mean(), b[col].mean()
        return f'{(av - bv) / av * 100:+.1f}%' if av > 0 else 'N/A'

    delta_rows.append({
        'provider':          PROVIDER_LABELS[p],
        'fix_success_delta': f'{(b["fix_success"].mean() - a["fix_success"].mean()):+.0%}',
        'token_reduction':   pct('total_tokens'),
        'latency_reduction': pct('latency'),
        'tool_reduction':    pct('tool_calls'),
        'attempt_reduction': pct('fix_attempts'),
    })

delta_df = pd.DataFrame(delta_rows).set_index('provider')
delta_df.columns = ['Fix Success Δ', 'Token Δ', 'Latency Δ', 'Tool Calls Δ', 'Fix Attempts Δ']
print('Positive = B (TraceLog) is better than A (Standard Log)')
delta_df

## Charts: B vs A Efficiency by Provider

In [ ]:
metrics = [
    ('tool_calls',   'Tool Calls',   'Number of tool calls'),
    ('latency',      'Latency (s)',  'Wall-clock time (seconds)'),
    ('total_tokens', 'Total Tokens', 'Input + output tokens'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Condition A (Standard Log) vs Condition B (TraceLog) — by Provider\ngpt-5.4 · claude-sonnet-4-6 · gemini-2.5-pro', fontsize=12, fontweight='bold', y=1.02)

x = np.arange(len(PROVIDERS))
width = 0.35

for ax, (col, title, ylabel) in zip(axes, metrics):
    vals_a = [df[(df['provider'] == p) & (df['condition'] == 'A')][col].mean() for p in PROVIDERS]
    vals_b = [df[(df['provider'] == p) & (df['condition'] == 'B')][col].mean() for p in PROVIDERS]

    bars_a = ax.bar(x - width/2, vals_a, width, label='A (Standard)', color=COLOR_A, alpha=0.85)
    bars_b = ax.bar(x + width/2, vals_b, width, label='B (TraceLog)', color=COLOR_B, alpha=0.85)

    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels([PROVIDER_LABELS[p].split()[0] for p in PROVIDERS])
    ax.legend(fontsize=9)
    ax.bar_label(bars_a, fmt='%.0f', padding=2, fontsize=8)
    ax.bar_label(bars_b, fmt='%.0f', padding=2, fontsize=8)
    ax.set_ylim(0, max(max(vals_a), max(vals_b)) * 1.3)

plt.tight_layout()
plt.savefig('docs/eval/benchmark_v3.2/results/benchmark_v3.2_comparison.png', bbox_inches='tight')
plt.show()

## Per-Scenario Token Heatmap (B vs A Reduction %)

In [ ]:
heat_data = []
for s in ALL_SCENARIOS:
    row = []
    for p in PROVIDERS:
        sub = df[(df['scenario'] == s) & (df['provider'] == p)]
        if len(sub) == 0:
            row.append(np.nan)
            continue
        a_tok = sub[sub['condition'] == 'A']['total_tokens'].values[0]
        b_tok = sub[sub['condition'] == 'B']['total_tokens'].values[0]
        reduction = (a_tok - b_tok) / a_tok * 100 if a_tok > 0 else 0
        row.append(reduction)
    heat_data.append(row)

heat_df = pd.DataFrame(
    heat_data,
    index=ALL_SCENARIOS,
    columns=[PROVIDER_LABELS[p].split()[0] for p in PROVIDERS],
)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heat_df.values, cmap='RdYlGn', aspect='auto', vmin=-20, vmax=95)
ax.set_xticks(range(len(PROVIDERS)))
ax.set_xticklabels(heat_df.columns, fontsize=11)
ax.set_yticks(range(len(ALL_SCENARIOS)))
ax.set_yticklabels(ALL_SCENARIOS, fontsize=10)
ax.axhline(len(SCENARIOS_V2) - 0.5, color='black', linewidth=2, linestyle='--')
ax.text(2.55, len(SCENARIOS_V2) - 0.55, '← v3 new (multi-threading)', fontsize=8, va='bottom')

for i in range(len(ALL_SCENARIOS)):
    for j in range(len(PROVIDERS)):
        val = heat_df.values[i, j]
        if not np.isnan(val):
            label = f'{val:.0f}%' if val >= 0 else 'ERR'
            ax.text(j, i, label, ha='center', va='center', fontsize=9,
                    fontweight='bold', color='black')

plt.colorbar(im, ax=ax, label='Token Reduction B vs A (%)')
ax.set_title('Token Reduction: TraceLog (B) vs Standard Log (A)\nGreen = TraceLog more efficient', fontweight='bold')
plt.tight_layout()
plt.savefig('docs/eval/benchmark_v3.2/results/benchmark_v3.2_token_heatmap.png', bbox_inches='tight')
plt.show()

## Single-Process vs Multi-Threading

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Token / Latency / Tool Calls: Single-Process vs Multi-Threading', fontsize=12, fontweight='bold', y=1.02)

scenario_types  = ['single-process', 'multi-threading']
type_labels     = ['Single-Process\n(v2 scenarios)', 'Multi-Threading\n(v3 new)']

for ax, (col, title, ylabel) in zip(axes, metrics):
    for i, (stype, label) in enumerate(zip(scenario_types, type_labels)):
        sub    = df[df['scenario_type'] == stype]
        vals_a = sub[sub['condition'] == 'A'][col].mean()
        vals_b = sub[sub['condition'] == 'B'][col].mean()
        x_pos  = np.array([i * 2.5, i * 2.5 + 0.8])
        bars   = ax.bar(x_pos, [vals_a, vals_b], 0.7, color=[COLOR_A, COLOR_B], alpha=0.85)
        for bar, val in zip(bars, [vals_a, vals_b]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals_a, vals_b) * 0.02,
                    f'{val:.0f}', ha='center', fontsize=8, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xticks([0.4, 2.9])
    ax.set_xticklabels(type_labels, fontsize=9)

patch_a = mpatches.Patch(color=COLOR_A, alpha=0.85, label='A (Standard)')
patch_b = mpatches.Patch(color=COLOR_B, alpha=0.85, label='B (TraceLog)')
fig.legend(handles=[patch_a, patch_b], loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

print('Token reduction B vs A:')
for stype, label in zip(scenario_types, ['Single-Process', 'Multi-Threading']):
    sub   = df[df['scenario_type'] == stype]
    a_tok = sub[sub['condition'] == 'A']['total_tokens'].mean()
    b_tok = sub[sub['condition'] == 'B']['total_tokens'].mean()
    delta = (a_tok - b_tok) / a_tok * 100
    print(f'  {label}: A={a_tok:.0f}  B={b_tok:.0f}  reduction={delta:+.1f}%')

## Key Findings

### 1. Fix Success Rate — All providers 100% on Standard Log
Unlike v3 (gpt-4o missed 2 scenarios on Standard Log), all v3.2 models fix every scenario in Condition A. Newer models are more capable on Standard Log alone. One exception: Google producer_aggregator B (GraphRecursionError, consistent across 2 runs).

### 2. Token Efficiency — Consistent reduction, Google most dramatic
TraceLog reduces token usage across all providers:
- **OpenAI (gpt-5.4)**: −28% (16k → 11.5k avg)
- **Anthropic (claude-sonnet-4-6)**: −22% (20k → 15.8k avg)
- **Google (gemini-2.5-pro)**: −80% (96k → 18.9k avg)

Google's high average in Condition A is driven by extreme outliers: `ledger_processor` (324k tokens) and `worker_dispatch` (217k tokens) without TraceLog. With TraceLog, Google drops to the same range as other providers.

### 3. Most Striking Case: Google ledger_processor
Standard Log: **324k tokens / 265s** → TraceLog: **22k tokens / 59s** (−93% tokens, −78% latency).
Gemini without TraceLog performs extensive exploration on complex multi-threading bugs. TraceLog's call path immediately surfaces the causal chain, collapsing the search space.

### 4. v3.2 vs v3 Comparison
| Metric | v3 OpenAI (gpt-4o, B) | v3.2 OpenAI (gpt-5.4, B) |
|---|---|---|
| Fix rate (A) | 71% | **100%** |
| Avg tokens (B) | ~14k | 11.5k |
| Avg latency (B) | ~23s | 18.3s |

gpt-5.4 is more capable and more token-efficient than gpt-4o. The fix success gap that TraceLog closed in v3 (71%→100%) no longer exists in v3.2 — both conditions pass — but TraceLog still reduces cost and latency.

### 5. Root Cause Identification — 0% across all
Consistent with v3: agents fix the bug but do not reliably name the root cause function. Judge prompt constraint, not model capability.

---

### Conclusion
> **TraceLog consistently reduces token cost and latency across all three providers and both single-process and multi-threading scenarios.** With newer models (v3.2), the fix success gap between conditions has closed — both Standard Log and TraceLog achieve high fix rates — but the efficiency advantage of TraceLog remains substantial, particularly on complex scenarios where standard logs provide no causal structure.